Modified Pöschl–Teller Potential

$$V(x) = - \frac{V_0}{\cosh^2(\alpha x)}
$$
where
$$E_n = -\frac{\hbar^2 \alpha^2}{2m}
\left[
\sqrt{\frac{2mV_0}{\hbar^2 \alpha^2} + \frac{1}{4}}
- \left( n + \tfrac{1}{2} \right)
\right]^2
$$

By substituting $\hbar=m=1$, $V_0 = 6, \alpha=0.5$ the expected results are
```
Expected : [-5.194, -3.708, -2.471,...]
```
with respect to `n = [0, 1, 2,...]`

### Setup

In [1]:
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import brentq

m, hbar = 1.0, 1.0

def V(x):
    V0 = 6
    a = 0.5
    return - V0/(np.cosh(a*x))**2

def func(x, u_vec, E):
    u, du = u_vec
    d2u = (2.0 * m / hbar**2) * (V(x) - E) * u
    return [du, d2u]

def shoot(E):
    sol = solve_ivp(fun=func, t_span=(-L, L), y0=[0, 1], method="RK45", args=(E,), atol=1e-8, rtol=1e-8)

    # u_final = u(L)
    u_final = sol.y[0, -1]

    # if |u_final|>1e10, return u_final to np.sign(u_final) * 1e10 to avoid overflow
    return np.sign(u_final) * min(abs(u_final), 1e10)

### Find optimal L
We guess
```
E_min, E_max = -10, 10
```
We defined `find_optimal()` and find $L_{optimal}$ with

```
for L in np.arange(2, 10, 1):
...
```

In [2]:
E_min, E_max = -10, 10

# Defind find_optimalL
def find_optimalL(E_min, E_max, tol=1e-6):
    print(f"{'L':<10} | {'E_root'}")
    print("-" * 25)
    E_prev = 0.0

#  Change L_min, L_max, and nsteps at here eg. Lmin = 2, Lmax=10, nsteps = 1
    for L in np.arange(1, 15, 1):

        def shoot(E):
            sol = solve_ivp(fun=func, t_span=(-L, L), y0=[0, 1], method="RK45", args=(E,), atol=1e-8, rtol=1e-8)

            u_final = sol.y[0, -1]

            return np.sign(u_final) * min(abs(u_final), 1e10)

        energies = np.linspace(E_min, E_max, 100)
        eigenvalues=[]
        f_old = shoot(energies[0])

        for i in range(len(energies) - 1):
            E_low = energies[i]
            E_high = energies[i+1]
            f_new = shoot(E_high)

            if f_old * f_new < 0:
                try:
                    E_root = brentq(shoot, E_low, E_high)
                    eigenvalues.append(E_root)
                    break
                except ValueError:
                    pass

            f_old = f_new

        if not eigenvalues:
            print(f"{L:<10.1f} | No eigenvalues found")
            continue

        E_root = eigenvalues[0]
        print(f"{L:<10.1f} | {E_root:.8f}")

        if abs(E_root - E_prev) < tol:
            print("-" * 25)
            print(f"Stability reached at L = {L:.1f}")

            return L, E_root
        E_prev = E_root

# Execute to find optimal L
optimal_L, stable_E = find_optimalL(E_min, E_max, tol=1e-6)

L          | E_root
-------------------------
1.0        | -4.58436918
2.0        | -5.18216337
3.0        | -5.19415716
4.0        | -5.19422209
5.0        | -5.19422225
-------------------------
Stability reached at L = 5.0


> Output : `L = 5.0 and E0 > - 6`

Thence, we define and run `find_eigenvalues()` with L = 5 or 6, \

We modify `E_min, E_max = -6, 10` to speed up

In [3]:
L = 5.0
E_min, E_max = -6, 10
# Define find_eigenvalues()
def find_eigenvalues(E_min, E_max, n, shoot=shoot):
    energies = np.linspace(E_min, E_max, n)
    eigenvalues = []

    f_old = shoot(energies[0])

    for i in range(len(energies) - 1):
        E_low = energies[i]
        E_high = energies[i+1]
        f_new = shoot(E_high)

        '''Sign change detected -> root found in this interval'''
        if f_old * f_new < 0:
            E_root = brentq(shoot, E_low, E_high)
            eigenvalues.append(E_root)

        f_old = f_new

    print(eigenvalues)

    if not eigenvalues:
        print("No eigenvalues found. Try different L or E_min, E_max.")

# Execute find_eigenvalues()
find_eigenvalues(E_min, E_max, n=100)

[-5.194222250532353, -3.70766664790057, -2.471098759913225, -1.483917102826765, -0.7347190023106924, -0.15244096557549808, 0.41170953558198076, 1.073428182274431, 1.8561950406064847, 2.753019844790945, 3.7586631726761106, 4.869999658638954, 6.0850392568403295, 7.402459722029288, 8.821350971020506]


When `L = 5.0` the eigenvalues give
```
[-5.194222250531993, -3.70766664790057, -2.4710987599131977, -1.4839171028267653, -0.7347190023107398,...]
```
Highly aligns what we expected : [-5.194, -3.708, -2.471,...]
